In [1]:
!set CMAKE_ARGS="-GGML_CUDA=on"
!set FORCE_CMAKE=1
!pip install llama-cpp-python -U \
--force-reinstall --no-cache-dir \
--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 GB 152.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 251.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 246.4 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: MarkupSafe
    Found existing installation: MarkupSafe 3.0.3
    Uninstalling MarkupSafe-3.0.3:
      Successfull

In [2]:
import sys
import os

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    os.system("cp -r /kaggle/input/datasets/lorenzozoccadelli/stag-hunt-env/Gymnasium-Stag-Hunt/gymnasium_stag_hunt /kaggle/working/")
    os.system("sed -i 's/from pettingzoo.utils import AgentSelector/from pettingzoo.utils import agent_selector as AgentSelector/g' /kaggle/working/gymnasium_stag_hunt/envs/pettingzoo/shared.py")
    
    sys.path.insert(0, '/kaggle/working/')

In [ ]:
import torch
from torch.distributions import Categorical
from huggingface_hub import login
import gc
from llama_cpp import Llama, llama_supports_gpu_offload
from llama_cpp.llama_chat_format import format_chatml
from abc import ABC, abstractmethod
import numpy as np
import random

import gymnasium as gym
from gymnasium.vector import SyncVectorEnv, AsyncVectorEnv, AutoresetMode
import gymnasium_stag_hunt
import time
import pickle
import matplotlib.pyplot as plt

SEED = 42

os.environ['PYTHONHASHSEED'] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

llama_supports_gpu_offload()

ggml_cuda_init: found 2 CUDA devices (Total VRAM: 29825 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB
  Device 1: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


True

# Utils

In [6]:
class RLAgent(ABC):
    
    @abstractmethod
    def policy(self, state):
        raise NotImplemented
    
    @abstractmethod
    def on_action_performed(self, state, action, reward, new_state, terminated, truncated) -> None:
        raise NotImplemented

    @abstractmethod
    def on_end(self):
        raise NotImplemented


In [7]:
import copy

def to_tensor(x):
    if not torch.is_tensor(x):
        x = torch.tensor(x, dtype=torch.float32)
    return x

def state_to_tensor(x):
    if not torch.is_tensor(x):
        x = torch.tensor(x, dtype=torch.float32)
    return x

def state_to_np(state):
    return np.array(state)

def arr_avg_last_n(arr, n=25, ignore_incompletes=False):
    arr = np.array(arr)
    arr_avgs = np.ndarray((len(arr),))
    for i in range(len(arr)):
        if ignore_incompletes and i<n-1:
            arr_avgs[i] = np.nan
        else:
            arr_avgs[i] = np.mean(arr[np.max([0, i-n+1]):i+1])
    
    return arr_avgs


'''
Problem: gym expects reward to be a scalar in vectorized environments
I create the wrapper which returns a mock 0.0 reward and store the real reward in the info
'''
class StagHuntEnvsWrapper(gym.Wrapper):
    def __init__(self, env, print_rewards=False, test=False):
        super().__init__(env)

        self.print_rewards = print_rewards
        self.total_reward = np.array([0.0, 0.0])
    
    def step(self, action):
        new_observation, rewards, termination, truncation, info = self.env.step(action)


        info["rewards"] = np.array(rewards)
        if self.print_rewards:
            print(info["rewards"])

        for i in range(2):
            self.total_reward[i] += rewards[i]
        
        if termination or truncation:
            info["total_rewards"] = self.total_reward
            if self.print_rewards:
                print(f"Episode cumulative reward: {self.total_reward}")

            self.total_reward = np.array([0.0, 0.0])

        return new_observation, 0.0, termination, truncation, info


class StagHuntEnvsLooper(ABC):

    def __init__(self, parallelization="sync", print_rewards=False, test=False):
        assert parallelization in ["sync", "async"]
        self.print_rewards = print_rewards
        self.test=test
        self.parallelization = parallelization
        self.evaluation_envs = None

    def load_default_env_creation_fns(self, env_id, env_params, wrapper):
        self.n_envs = 1
        self.envs_creation_fns = [lambda: wrapper(gym.make(env_id, **env_params), self.print_rewards, self.test)]
        self.create_envs()

    def create_envs(self):
        if self.parallelization=="sync":
            self.envs = SyncVectorEnv(self.envs_creation_fns, autoreset_mode=AutoresetMode.SAME_STEP)
        else:
            self.envs = AsyncVectorEnv(self.envs_creation_fns, autoreset_mode=AutoresetMode.SAME_STEP)

    def get_action_space(self):
        return self.envs.action_space

    @abstractmethod
    def on_episode_end(self, info, terminations, truncations):
        return

    @abstractmethod
    def on_evaluation_episode_end(self, info, terminations, truncations, already_ended):
        return
    
    @abstractmethod
    def convert_state(self, observation, n_envs, info):
        raise NotImplemented

    def extract_rewards_from_info(self, info, new_observations, n_envs):
        if "_final_obs" in info:
            
            next_iter_observations = copy.deepcopy(new_observations)
            for i in np.where(info["_final_obs"])[0]:
                new_observations[i] = info["final_obs"][i]
            new_observations = self.convert_state(new_observations, n_envs, info)
            next_iter_observations = self.convert_state(next_iter_observations, n_envs, info)

            assert "_final_info" in info
            rewards = np.zeros((n_envs, 2), dtype=np.float32)
            rewards[info["final_info"]["_rewards"]] = info["final_info"]["rewards"]

            if "_rewards" in info:
                rewards[info["_rewards"]] = info["rewards"]
            
            rewards = rewards.T

        else:
            new_observations = self.convert_state(new_observations, n_envs, info)
            next_iter_observations = new_observations
            rewards = info["rewards"].T
        
        return rewards, new_observations, next_iter_observations

    def training_loop(self, agent1: RLAgent, agent2: RLAgent, max_steps, common_reward=False, enable_gui=False, printing_delay=1, ts_to_print=1000):

        assert self.n_envs==1 or (self.n_envs>1 and not enable_gui)

        start_time = time.perf_counter()
        steps = 0
        observations, info = self.envs.reset()
        observations = self.convert_state(observations, self.n_envs, info)

        while True:

            ### Policy Section
            actions_agent1, actions_agent2 = agent1.policy(observations[0][0]), agent2.policy(observations[0][1])

            ### Step Section
            if enable_gui:
                time.sleep(printing_delay)
                self.envs.render()

            new_observations, rewards, terminations, truncations, info = self.envs.step(np.stack([actions_agent1, actions_agent2]).T)

            rewards, new_observations, next_iter_observations = self.extract_rewards_from_info(info, new_observations, self.n_envs)

            
            reward_agent1 = rewards[0] + rewards[1] if common_reward else rewards[0]
            reward_agent2 = rewards[0] + rewards[1] if common_reward else rewards[1]

            ### Give the step to the agents
            agent1.on_action_performed(observations[0][0], actions_agent1, reward_agent1, new_observations[0][0], terminations, truncations)
            agent2.on_action_performed(observations[0][1], actions_agent2, reward_agent2, new_observations[0][1], terminations, truncations)

            ### Step End section
            if steps !=0 and steps%ts_to_print == 0:
                print(f"Step {steps}/{max_steps}")


            steps+=1

        
            ### Episode End section
            if (terminations | truncations).any():
                self.on_episode_end(info, terminations, truncations)

            if steps>=max_steps:
                break

            observations = next_iter_observations

        end_time = time.perf_counter()
        print(f"Training lasted for {end_time - start_time:.2f}s")
    


# DQN

In [ ]:

class PrioritizedExperienceReplayBuffer:
    def __init__(self, state_shape, dim, start_after=None, alpha=0.6, beta=0.4, beta_iters=100000, epsilon=1e-6):
        self.dim = dim
        
        self.states = torch.zeros((dim, *state_shape), dtype=torch.float32)
        self.actions = torch.zeros(dim, dtype=torch.int64)
        self.rewards = torch.zeros(dim, dtype=torch.float32)
        self.next_states = torch.zeros((dim, *state_shape), dtype=torch.float32)
        self.terminals = torch.zeros(dim, dtype=torch.float32)
        self.priorities = torch.zeros(dim, dtype=torch.float32)

        self.alpha = alpha
        self.beta = beta
        self.beta_increment = (1.0 - beta) / beta_iters
        self.max_priority = 1.0
        self.epsilon = epsilon

        self.current_idx = 0
        self.full = False
        self.total_full = False
        if start_after==None:
            self.start_after = dim
        else:
            self.start_after = start_after
    
    def add_experience(self, state, action, reward, next_state, terminal):
        self.states[self.current_idx] = torch.as_tensor(state, dtype=torch.float32)
        self.actions[self.current_idx] = torch.as_tensor(action, dtype=torch.int64)
        self.rewards[self.current_idx] = torch.as_tensor(reward, dtype=torch.float32)
        self.next_states[self.current_idx] = torch.as_tensor(next_state, dtype=torch.float32)
        self.terminals[self.current_idx] = torch.as_tensor(terminal, dtype=torch.float32)
        self.priorities[self.current_idx] = self.max_priority

        self.current_idx = (self.current_idx + 1) % self.dim

        if not self.full and (self.current_idx == 0 or self.current_idx >= self.start_after):
            self.full = True
        
        if not self.total_full and self.current_idx == 0:
            self.total_full = True
    
    def sample_batch(self, batch_size):
        if not self.full:
            raise ValueError("Il buffer non è ancora pieno!")

        dim = self.dim if self.total_full else self.current_idx
        prios = self.priorities[:dim]

        probs = prios ** self.alpha
        probs /= probs.sum()

        indices = torch.multinomial(probs, batch_size, replacement=True)

        weights = (dim * probs[indices]) ** (-self.beta)
        weights /= weights.max()
        
        self.beta = min(1.0, self.beta + self.beta_increment)


        b_states = self.states[indices]
        b_next_states = self.next_states[indices]
        b_actions = self.actions[indices, None]
        b_rewards = self.rewards[indices, None]
        b_terminals = self.terminals[indices, None]

        return b_states, b_actions, b_rewards, b_next_states, b_terminals, indices, weights


    def update_priorities(self, indices, td_errors):
        new_priorities = (torch.abs(td_errors) + self.epsilon).cpu()
        self.priorities[indices] = new_priorities
        self.max_priority = max(self.max_priority, new_priorities.max().item())



In [ ]:
import torch.nn.functional as F

 
class DuelingDDQNAgent:
    def __init__(self, dqn, er_buffer: PrioritizedExperienceReplayBuffer, width, height, n_plants, action_space, lr=1e-4, batch_size=32, start_epsilon=1.0, 
                 epsilon_decay=0.99, min_epsilon=0.05, gamma=0.99, device="cpu", test=False, linear_decay=False, update_every_n_steps=1, n_envs=1, target_update_freq=1000,
                 use_lr_scheduler=False, end_lr=1e-7, lr_decay_steps=100000):
        
        self.dqn = dqn
        self.lr = lr
        self.optim = torch.optim.Adam(dqn.parameters(), lr=lr)

        self.target_dqn = copy.deepcopy(self.dqn)
        self.target_update_freq = target_update_freq
        self.total_steps = 0

        self.gamma = gamma
        self.action_space = action_space

        self.er_buffer = er_buffer
        self.batch_size = batch_size

        self.start_epsilon = start_epsilon
        self.epsilon = start_epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        self.device = device
        self.dqn = self.dqn.to(device)
        self.target_dqn = self.target_dqn.to(device)
        self.test = test
        self.linear_decay = linear_decay
        self.update_every_n_steps = update_every_n_steps
        self.step_counter = 0
        self.n_envs = n_envs

        self.width, self.height, self.n_plants =  width, height, n_plants

        self.use_lr_scheduler = use_lr_scheduler
        self.end_lr = end_lr
        self.lr_decay_steps = lr_decay_steps
        if use_lr_scheduler:
            self.lr_scheduler = torch.optim.lr_scheduler.LinearLR(self.optim, start_factor=1.0, end_factor=end_lr/lr, total_iters=lr_decay_steps)

    def policy(self, state) -> int:
        state = self.convert_state(state)
        state = state.to(self.device)
        with torch.no_grad():
            actions = self.dqn(state).argmax(dim=1)
        
        if not self.test:
            epsilon_probs = torch.rand(self.n_envs, device=self.device)
            n_actions = self.action_space.nvec[0] if hasattr(self.action_space, 'nvec') else self.action_space.n
            random_actions = torch.randint(0, n_actions, size=(self.n_envs,), device=self.device)
            actions = torch.where(epsilon_probs < self.epsilon, random_actions, actions)

        return actions.cpu().numpy()

    def save_new_experience(self, state, action, reward, new_state, terminated, truncated):
        state = self.convert_state(state)
        new_state = self.convert_state(new_state)

        state, new_state = to_tensor(state), to_tensor(new_state)
        for i in range(self.n_envs):
            self.er_buffer.add_experience(
                state[i],
                action[i],
                reward[i],
                new_state[i],
                1 if terminated[i] or truncated[i] else 0
            )
  
    def update_network(self):
        self.step_counter = 0
 
        batch = self.er_buffer.sample_batch(self.batch_size)

        states, actions, rewards, next_states, terminals, indices, weights = batch

        states, next_states = states.to(self.device), next_states.to(self.device)
        actions, rewards, terminals = actions.to(self.device), rewards.to(self.device), terminals.to(self.device)

        q = torch.gather(self.dqn(states), dim=1, index=actions)

       
        with torch.no_grad():
            best_next_actions = self.dqn(next_states).argmax(dim=1, keepdim=True)
            target_next_qs = self.target_dqn(next_states)
            next_q = torch.gather(target_next_qs, dim=1, index=best_next_actions)

        target = rewards + self.gamma * next_q * (1 - terminals)
        

        td_errors = target.squeeze() - q.squeeze()
        self.er_buffer.update_priorities(indices, td_errors.detach())

        loss = (weights * (td_errors ** 2)).mean()

        self.optim.zero_grad()
        loss.backward()
        self.optim.step()  
        
        if self.use_lr_scheduler:
            self.lr_scheduler.step()

    def on_action_performed(self, state, action, reward, new_state, terminated, truncated) -> None:
        if self.test:
            return

        self.save_new_experience(state, action, reward, new_state, terminated, truncated)
        self.update_epsilon()

        self.step_counter += 1
        self.total_steps += 1

        if self.total_steps % self.target_update_freq == 0:
            self.target_dqn.load_state_dict(self.dqn.state_dict())

        if self.step_counter < self.update_every_n_steps:
            return

        if not self.er_buffer.full:
            return

        self.update_network()

    def update_epsilon(self) -> None:
        if not self.test:
            if self.linear_decay:
                self.epsilon = max(self.min_epsilon, self.epsilon - self.epsilon_decay)
            else:
                self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

    def convert_state(self, states):
        new_states = torch.zeros((1, 4+2*int(self.n_plants)), dtype=torch.float32)

        new_states[0, 0] = (int(states[2])-int(states[0]))/self.width
        new_states[0, 1] = (int(states[3])-int(states[1]))/self.height
        new_states[0, 2] = (int(states[4])-int(states[0]))/self.width
        new_states[0, 3] = (int(states[5])-int(states[1]))/self.height
        for i in range(self.n_plants):
            new_states[0, 4+2*i] = (int(states[6+2*i]) - int(states[0]))/self.width
            new_states[0, 4+2*i+1] = (int(states[6+2*i+1]) - int(states[1]))/self.height
            
        return new_states


    

# LLM Stag Hunt Utils

In [ ]:

class StagHuntLog:
    def __init__(self, n_envs):
        self.episode_rewards = [[] for _ in range(n_envs)]
        self.episode_foragings = [[] for _ in range(n_envs)]
        self.episode_stags = [[] for _ in range(n_envs)]
        self.episode_maulings = [[] for _ in range(n_envs)]
        self.hyperparameters = {}

    def save(self, file_name):
        with open(file_name, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, file_name):
        with open(file_name, "rb") as f:
            log = pickle.load(f)
        return log

    def get_plot_data(self, episodes_to_mean=25):

        tot_rewards = np.array(self.episode_rewards).mean(axis=0)
        foragings = np.array(self.episode_foragings).mean(axis=0)
        stags = np.array(self.episode_stags).mean(axis=0)
        maulings = np.array(self.episode_maulings).mean(axis=0)

        # In the form of (title, data, trend, x_label, y_label)
        data = [
            ("Rewards", tot_rewards, arr_avg_last_n(tot_rewards, episodes_to_mean), "Episode", "Tot. Reward"),
            ("Foragings", foragings, arr_avg_last_n(foragings, episodes_to_mean), "Episode", "N. Foragins"),
            ("Stags", stags, arr_avg_last_n(stags, episodes_to_mean), "Episode", "N. Stags"),
            ("Maulings", maulings, arr_avg_last_n(maulings, episodes_to_mean), "Episode", "N. maulings")
        ]

        return data

    def plot_stats(self, file_name, fig_n=1, plot_layout=(2, 2), episodes_to_mean=100):

        plt.figure(fig_n, figsize=(20, 10))

        plt.subplot(*plot_layout, 1)
        plt.title("Agent 1 (blue) Rewards")
        plt.xlabel("Episode")
        plt.ylabel("Tot. Reward")
        plt.plot(self.episode_rewards[0])
        plt.plot(arr_avg_last_n(self.episode_rewards[0], episodes_to_mean), color="red")

        plt.subplot(*plot_layout, 2)
        plt.title("Agent 1 (blue) Foragings")
        plt.xlabel("Episode")
        plt.ylabel("N. Foragings")
        plt.plot(self.episode_foragings[0])
        plt.plot(arr_avg_last_n(self.episode_foragings[0], episodes_to_mean), color="red")

        plt.subplot(*plot_layout, 3)
        plt.title("Agent 1 (blue) Stags")
        plt.xlabel("Episode")
        plt.ylabel("N. Stags")
        plt.plot(self.episode_stags[0])
        plt.plot(arr_avg_last_n(self.episode_stags[0], episodes_to_mean), color="red")

        plt.subplot(*plot_layout, 4)
        plt.title("Agent 1 (blue) Maulings")
        plt.xlabel("Episode")
        plt.ylabel("N. Maulings")
        plt.plot(self.episode_maulings[0])
        plt.plot(arr_avg_last_n(self.episode_maulings[0], episodes_to_mean), color="red")

        plt.savefig(file_name)


class StagHuntWrapper(StagHuntEnvsWrapper):
    def __init__(self, env, print_rewards=False, test=False):
        super().__init__(env, print_rewards=print_rewards)
        
        self.total_foragings = np.array([0, 0])
        self.total_stags = np.array([0, 0])
        self.total_maulings = np.array([0, 0])

    def step(self, action):
        new_observation, rewards, termination, truncation, info = super().step(action)
        for i in range(2):
            if self.unwrapped.forage_reward == info["rewards"][i]:
                self.total_foragings[i]+=1
            if self.unwrapped.stag_reward == info["rewards"][i]:
                self.total_stags[i]+=1
            if self.unwrapped.mauling_punishment == info["rewards"][i]:
                self.total_maulings[i]+=1

        if termination or truncation:
            info["total_foragings"] = self.total_foragings
            info["total_stags"] = self.total_stags
            info["total_maulings"] = self.total_maulings

            self.total_foragings = np.array([0, 0])
            self.total_stags = np.array([0, 0])
            self.total_maulings = np.array([0, 0])

        return new_observation, rewards, termination, truncation, info

class LLMStagHuntLooper(StagHuntEnvsLooper):
    def __init__(self, env_params, print_rewards=False):
        super().__init__("sync", print_rewards, False)

        self.load_default_env_creation_fns("StagHunt-Hunt-v0", env_params, StagHuntWrapper)

        self.log_agents = [StagHuntLog(1), StagHuntLog(1)]

        self.n_plants = env_params["forage_quantity"]
        self.width, self.height = env_params["grid_size"][0], env_params["grid_size"][1]

    def update_logs(self, logs_arr, mask, info):
        for i in np.where(mask)[0]:
            for j in range(2):
                logs_arr[j].episode_rewards[i].append(info["final_info"]["total_rewards"][i, j])
                logs_arr[j].episode_foragings[i].append(info["final_info"]["total_foragings"][i, j])
                logs_arr[j].episode_maulings[i].append(info["final_info"]["total_maulings"][i, j])
                logs_arr[j].episode_stags[i].append(info["final_info"]["total_stags"][i, j])

    def on_episode_end(self, info, terminations, truncations):
        endeds = truncations | terminations
        if endeds.any():
            self.update_logs(self.log_agents, endeds, info)
    
    def on_evaluation_episode_end(self, info, terminations, truncations, already_ended):
        endeds = truncations | terminations
        endeds[already_ended] = False
        if endeds.any():
            self.update_logs(self.evaluation_log_agents, endeds, info)

    def convert_state(self, states, n_envs, info):
        return states


# LLM Agent

In [11]:
from collections import defaultdict

def state_to_str(state, width, height):
    x_A, y_A, x_B, y_B, x_stag, y_stag, x_p1, y_p1, x_p2, y_p2 = state

    cells = defaultdict(set)
    cells[(x_A, y_A)].add("A")
    cells[(x_B, y_B)].add("B")
    cells[(x_stag, y_stag)].add("S")
    cells[(x_p1, y_p1)].add("P")
    cells[(x_p2, y_p2)].add("P")

    s = ""

    
    for y in range(height):
        row_str = "|"
        for x in range(width):
            content = cells[(x, y)]
            
            to_add = ""
            if not content:
                to_add += "." 
            else:
                if "A" in content:
                    to_add += "A"
                if "B" in content:
                    to_add += "B"
                if "S" in content:
                    to_add += "S"
                if "P" in content:
                    to_add += "P"  
            
            if len(to_add)==1:
                row_str+=f" {to_add} "
            if len(to_add)==2:
                row_str+=f"{to_add} "
        row_str += "|"
        s+=row_str+"\n"

    return s

class LLMStagHuntLog:
    def __init__(self, width, height, ask_for_cot=True, ask_for_critique=False):
        self.episode_log = []
        self.width = width
        self.height = height
        self.ask_for_critique = ask_for_critique
        self.ask_for_cot = ask_for_cot
    
    def add_log(self, step_log):
        self.episode_log.append({
            "state": step_log["state"],
            "state_description": step_log["state_description"],
            "decision": step_log["decision"],
            "reward": step_log["reward"]
        })

        if self.ask_for_cot:
            self.episode_log[-1]["cot"] = step_log["cot"]

        if self.ask_for_critique:
            self.episode_log[-1]["critique"] = step_log["critique"]
            self.episode_log[-1]["revised_decision"] = step_log["revised_decision"]
    
    def save(self, file_name):
        with open(file_name, "wb") as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, file_name):
        with open(file_name, "rb") as f:
            log = pickle.load(f)
        return log

    def dump(self):
        s=""
        for i, log in enumerate(self.episode_log):
            s+=f"Step: {i+1}\n"
            s+=f"{state_to_str(log["state"], self.width, self.height)}\n\n"
            s+=f"State description: {log["state_description"]}\n\n"
            if self.ask_for_cot:
                s+=f"CoT: {log["cot"]}\n"
            s+=f"Decision: {log["decision"]}\n"
            
            if self.ask_for_critique:
                s+=f"\nCritique: {log["critique"]}\n"
                s+=f"Revised Decision: {log["revised_decision"]}\n"

            s+=f"Reward: {log["reward"]}\n\n"
        return s

class LLMStagHuntAgent(RLAgent):
    def __init__(self, model, width, height, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False):
        self.model = model

        self.width, self.height = width, height
        self.agent_log = LLMStagHuntLog(width, height, ask_for_cot, ask_for_critique)
        self.current_log = {}
        self.biasing_string = biasing_string

        assert not ask_for_critique or (ask_for_critique and ask_for_cot)
        self.ask_for_cot = ask_for_cot
        self.ask_for_critique = ask_for_critique
        self.enable_think_tag = enable_think_tag

        if not self.ask_for_cot:
            self.setup_no_cot_base_conversation()
        if self.ask_for_cot and not self.ask_for_critique:
            self.setup_base_conversation()
        if self.ask_for_critique:
            self.setup_critique_base_conversation()

        

    def get_env_description(self):
        return f"""You are Agent A playing Stag Hunt on a {self.width}x{self.height} grid. 
Your goal is to maximize points moving around the grid. You can either earn or lose points in these ways:
- Catching the Stag with Agent B = +5 points. You and Agent B must be in the same cell and the Stag must move toward you to catch it.
- Eating a Plant = +1 point. To eat a plant you just need to step into it.
- Hit by Stag while Agent B is NOT in your cell = -0.5 points. You're hit by the Stag if the Stag moves into your cell and Agent B is NOT in your same cell.

Some other mechanisms of the environment:
- The Stag always runs 1 step toward the nearest agent. To catch the stag you, B and the Stag must be in the same cell after the move of everyone.
- If Agent B and the Stag are in the same cell BUT you are in a different cell, the stag is NOT caught and Agent B takes the -0.5 penelaty.
- If you are in the same cell of the Stag BUT Agent B is NOT in your same cell, the Stag is NOT caught and you recieve the -0.5 penalty.
- If A and B are in the same cell and the Stag is only 1 step far from you, if both of you stay still (idle) the Stag will come over you in the next turn and you will catch it.

Allowed actions for Agent A (you):
- U (Up): Move one step up
- D (Down): Move one step down
- L (Left): Move one step left
- R (Right): Move one step right
- I (Idle): Don't move and stay in the same cell

{self.biasing_string}
"""
    
    def setup_critique_base_conversation(self):
        test_state1 = [0, 0, 2, 2, 2, 1, 1, 2, 2, 1]
        test_state2 = [1, 1, 1, 1, 2, 1, 2, 2, 0, 0]
        test_state3 = [self.width-1, 2, self.width-1, 2, self.width-1, 0, 0, 0, 1, 1]

        self.critique_request_string = "Critique your last reasoning and decision. Your last reasoning and decision can be right, wrong or discrete but improvable, so don't be biased, be as fair as possible. Point out if another possible decision could improve the chances of an higher score and if another decision could be better to cooperate with agent B. You can either change your old decision or confirm it."

        self.base_conversation = [
        {
            "role": "system",
            "content": f"""{self.get_env_description()}
Analyze the state step-by-step, then output your final action. Your response must STRICTLY follow this format:
Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}Your thoughts or/and draft. Be concise and do NOT repeat yourself. End the reasoning exactly once when ready.{"[/THINK]" if self.enable_think_tag else ""}
Decision: <your action letter>

If the user asks for a Critique your response to the critique request must STRICTLY follow this format:
Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}Your critique at the previous response. Be as fair as possible pointing out either if the last response was good or not. Be concise and do NOT repeat yourself. End the reasoning exactly once when ready.{"[/THINK]" if self.enable_think_tag else ""}
Decision: <your action letter>
"""
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state1)}"
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}I am restricted from moving Left or Up, placing me at the top-left edge of my available grid. The Stag is 3 moves away (2 Right, 1 Down), while Agent B is 4 moves away (2 Right, 2 Down). The Stag is much closer to Agent B (only 1 step away) than it is to me, meaning the Stag will likely move Down towards Agent B on this turn. Since I am too far away to catch the Stag right now, my best strategy is to close the distance to where the action is. Plant 2 is in the same cell as the Stag's current position. Moving Right brings me closer to the Stag, Agent B, and Plant 2, setting me up for a coordinated catch or a plant grab in future turns.{"[/THINK]" if self.enable_think_tag else ""}\nDecision:R"
        },
        {
            "role": "user",
            "content": self.critique_request_string
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}The spatial reasoning is quite good. But the Stag is going to move toward B for shure. Anyway it does make sense to move toward B to join it for a future catch. It's good also thinking about a possible collection of a plant, even though the priority is still cooperate with B to get the Stag.{"[/THINK]" if self.enable_think_tag else ""}\nDecision: R"
        },


        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state2)}"
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}Agent B and I are currently sharing the exact same cell. The Stag is right next to us, just 1 step Right. In the next turn the Stag will be in our cell. Because the Stag is only 1 step away, I must move Right to enter its cell and initiate the joint catch alongside B.{"[/THINK]" if self.enable_think_tag else ""}\nDecision: R"
        },
        {
            "role": "user",
            "content": self.critique_request_string
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}The reasoning about the space and the distances is good. The idea to coordinate with B to catch the Stag is the ideal choice. But it's wrong saying that if I move Right I'll get the Stag because in that case I actually will switch with the stag missing the capture. Moreover if B stay still instead, it will be hit alone by the Stag, providing it losing -0.5 points. So it's better wait for the stag to come toward us staying Idle for this turn.{"[/THINK]" if self.enable_think_tag else ""}\nDecision: I"
        },


        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state3)}"
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}This is a good setup. Both me and Agent B are currently occupying the exact same cell, which is just 2 step Up from the Stag. To catch the Stag, both Agent B and I need to make the Stag move toward us. The next move of the Stag will be Down since will move toward us. If we both stay in the same cell for 2 turns the Stag will move to us and we'll catch it, but by simply moving Up, if B cooperate and also moves Up, we will successfully catch the stag only after this move and earning the maximum +5 points. The plants are too far away to matter, and catching the Stag is the optimal priority.{"[/THINK]" if self.enable_think_tag else ""}\nDecision: U"
        },
        {
            "role": "user",
            "content": self.critique_request_string
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}The reasoning about the positions of the items in the grid seems fine. Defining the Stag as the priority is a good choice because plants are too far and the Stag earns a way higher reward. The decision to move and cooperate with B is excelent. Also the choice of moving Up to earn the +5 points already in the next turn is very good.{"[/THINK]" if self.enable_think_tag else ""}\nDecision: U"
        }
        ]

    def setup_base_conversation(self):
        test_state1 = [0, 0, 2, 2, 2, 1, 1, 2, 2, 1]
        test_state2 = [1, 1, 1, 1, 2, 1, 2, 2, 0, 0]
        test_state3 = [self.width-1, 2, self.width-1, 2, self.width-1, 0, 0, 0, 1, 1]

        self.base_conversation = [
        {
            "role": "system",
            "content": f"""{self.get_env_description()}
Analyze the state step-by-step, then output your final action. Your response must STRICTLY follow this format:
Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}Your thoughts or/and draft. Be concise and do NOT repeat yourself. End the reasoning exactly once when ready.{"[/THINK]" if self.enable_think_tag else ""}
Decision: <your action letter>

"""
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state1)}"
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}I am restricted from moving Left or Up, placing me at the top-left edge of my available grid. The Stag is 3 moves away (2 Right, 1 Down), while Agent B is 4 moves away (2 Right, 2 Down). The Stag is much closer to Agent B (only 1 step away) than it is to me, meaning the Stag will likely move Down towards Agent B on this turn. Since I am too far away to catch the Stag right now, my best strategy is to close the distance to where the action is. Plant 2 is in the same cell as the Stag's current position. Moving Right brings me closer to the Stag, Agent B, and Plant 2, setting me up for a coordinated catch or a plant grab in future turns.{"[/THINK]" if self.enable_think_tag else ""}\nDecision:R"
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state2)}"
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}Agent B and I are currently sharing the exact same cell. The Stag is right next to us, just 1 step Right. Because the Stag always moves 1 step toward the nearest agent—and both Agent B and I are tied as the closest entities—the Stag is guaranteed to move Left and step directly into our current cell. If I stay Idle, and Agent B also stays, the Stag will walk right into our trap, securing the Catch for +5 points. Moving anywhere else risks swapping places with the Stag or missing the joint catch.{"[/THINK]" if self.enable_think_tag else ""}\nDecision:I"
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state3)}"
        },
        {
            "role": "assistant",
            "content": f"Let's think step by step: {"[THINK]" if self.enable_think_tag else ""}This is a good setup. Both me and Agent B are currently occupying the exact same cell, which is just 2 step Up from the Stag. To catch the Stag, both Agent B and I need to make the Stag move toward us. The next move of the Stag will be Down since will move toward us. If we both stay in the same cell for 2 turns the Stag will move to us and we'll catch it, but by simply moving Up, if B cooperate and also moves Up, we will successfully catch the stag only after this move and earning the maximum +5 points. The plants are too far away to matter, and catching the Stag is the optimal priority.{"[/THINK]" if self.enable_think_tag else ""}\nDecision: U"
        }
        ]

    def setup_no_cot_base_conversation(self):
        test_state1 = [0, 0, 2, 2, 2, 1, 1, 2, 2, 1]
        test_state2 = [1, 1, 1, 1, 2, 1, 2, 2, 0, 0]
        test_state3 = [self.width-1, 2, self.width-1, 2, self.width-1, 0, 0, 0, 1, 1]

        self.base_conversation = [
        {
            "role": "system",
            "content": f"""{self.get_env_description()}
After you are told every state, output your action. Your response must STRICTLY follow this format:
Decision: <your action letter>
"""
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state1)}\nDecision: "
        },
        {
            "role": "assistant",
            "content": "R"
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state2)}\nDecision: "
        },
        {
            "role": "assistant",
            "content": "I"
        },
        {
            "role": "user",
            "content": f"State: {self.convert_state_to_prompt(test_state3)}\nDecision: "
        },
        {
            "role": "assistant",
            "content": "U"
        }
        ]




    def relative_dist_to_str(self, x1, y1, x2, y2):
        dx = int(x2)-int(x1)
        dy = int(y2)-int(y1)

        if dx==0 and dy==0:
            return "is in the same cell as you"
        
        first_part, second_part = "", ""
        if dx!=0:
            first_part = f"{abs(dx)} step{"s" if abs(dx)>1 else ""} {"Left" if dx<0 else "Right"}"
        
        if dy!=0:
            second_part = f"{abs(dy)} step{"s" if abs(dx)>1 else ""} {"Up" if dy<0 else "Down"}"

        return first_part + (" and " if first_part!="" and second_part!="" else "") + second_part + " to you"

    def block_string(self, x, y):
        first_part = ""
        second_part = ""
        if x==0:
            first_part = " You can't move L."
        if x==self.width-1:
            first_part = " You can't move R."
        if y==0:
            second_part = " You can't move U."
        if y==self.height-1:
            second_part = " You can't move D."

        return first_part+second_part

    def stag_distance_str(self, x, y, stag_x, stag_y):
        dist = abs(int(x)-int(stag_x)) + abs(int(y)-int(stag_y))

        return f"The Stag is in total {dist} steps distant from"

    def convert_state_to_prompt(self, state):
        s1 = self.relative_dist_to_str(state[0], state[1], state[2], state[3])
        s2 = self.relative_dist_to_str(state[0], state[1], state[4], state[5])
        s3 = self.relative_dist_to_str(state[0], state[1], state[6], state[7])
        s4 = self.relative_dist_to_str(state[0], state[1], state[8], state[9])
        as_dist_str = self.stag_distance_str(state[0], state[1], state[4], state[5])
        bs_dist_str = self.stag_distance_str(state[2], state[3], state[4], state[5])
        bs = self.block_string(state[0], state[1])

        reasoning_forcing_str = "Consider all this information to do your move. Try to think also to the possible move of the agent B and what is the best you can do to coordinate and cooperate with it. \
            Remember that to catch the stag you need to be togheter in the same cell when the stag step into your cell.\
            Play for your outcome, but in case of need you can trust the Agent B"

        str_state = f"B is {s1}. Stag is {s2}. Plant 1 is {s3}. Plant 2 is {s4}. {as_dist_str} you. {bs_dist_str} B. {bs}. {reasoning_forcing_str}"
        
        return str_state

    def convert_decision_to_action(self, decision):
        if decision=="L":
            return 0
        if decision=="D":
            return 1
        if decision=="R":
            return 2
        if decision=="U":
            return 3
        if decision=="I":
            return 4
        return np.random.randint(0, 5)




    def ask_model(self, conversation):
        resp = self.model.create_chat_completion(conversation)
        txt_resp = resp["choices"][0]["message"]["content"]
        return txt_resp
    
    def policy_no_cot(self, state_description):

        target_tokens = ["L", "R", "U", "D", "I"]
        target_ids = [self.model.tokenize(char.encode("utf-8"), add_bos=False)[0] for char in target_tokens]

        conversation=self.base_conversation+[
            {
                "role": "user",
                "content": f"State: {state_description}\nDecision: "
            },
            {
                "role": "assistant",
                "content": ""
            }
        ]

        prompt = format_chatml(messages=conversation).prompt
        input_tokens = self.model.tokenize(prompt.encode("utf-8"))

        self.model.reset()
        self.model.eval(input_tokens)
        next_token_logits = self.model.scores[-1]
        
        logits = torch.tensor([float(next_token_logits[t_id]) for t_id in target_ids])
        dist = Categorical(logits=logits)

        decision_txt = target_tokens[dist.sample().item()]
        self.current_log["decision"] = decision_txt
        return np.array([self.convert_decision_to_action(decision_txt)])

    def policy(self, state):
        
        state_description = self.convert_state_to_prompt(state)
        self.current_log["state"] = state
        self.current_log["state_description"] = state_description

        if not self.ask_for_cot:
            return self.policy_no_cot(state_description)

        prompt=self.base_conversation+[
            {
                "role": "user",
                "content": f"State: {state_description}"
            },
        ]

        txt_resp = self.ask_model(prompt)

        cot = txt_resp.split("Decision:")[0]
        decision_txt = txt_resp.split("Decision:")[-1].strip()

        
        
        self.current_log["cot"] = cot
        self.current_log["decision"] = decision_txt

        if self.ask_for_critique:
            prompt=prompt+[
                {
                    "role": "assistant",
                    "content": txt_resp
                },
                {
                    "role": "user",
                    "content": self.critique_request_string
                },
            ]

            txt_resp = self.ask_model(prompt)

            critique = txt_resp.split("Decision:")[0]
            decision_txt = txt_resp.split("Decision:")[-1].strip()

            self.current_log["critique"] = critique
            self.current_log["revised_decision"] = decision_txt
                

        return np.array([self.convert_decision_to_action(decision_txt)])
        
    
    def on_action_performed(self, state, action, reward, new_state, terminated, truncated) -> None:
        self.current_log["reward"] = reward[0]

        self.agent_log.add_log(self.current_log)
        self.current_log={}

    def on_end(self):
        pass

# Config

In [ ]:
import argparse
arg_parser = argparse.ArgumentParser()

# Generic
arg_parser.add_argument("-x", "--use-code-params", action="store_true", help="Use params embedded into the code")
arg_parser.add_argument("-t", "--test", action="store_true", help="Enables test mode")
arg_parser.add_argument("-s", "--save", action="store_true", help="Save weights or estimations")
arg_parser.add_argument("-c", "--checkpoint", action="store_true", help="Start from checkpoint")

arg_parser.add_argument("-bf", "--base-folder", type=str, default="savings/stag_hunt/", help="Stats file name")
arg_parser.add_argument("-n", "--name", type=str, default="stag_hunt", help="Generic name")

# Environment
arg_parser.add_argument("-E", "--n-envs", type=int, default=1, help="N of parallel environments")
arg_parser.add_argument("-W", "--width", type=int, default=4, help="Width of the grid")
arg_parser.add_argument("-H", "--height", type=int, default=4, help="Height of the grid")
arg_parser.add_argument("-S", "--time-steps", type=int, default=200, help="N of time steps in each episode")
arg_parser.add_argument("-e", "--episodes", type=int, default=1000, help="N of episodes")

arg_parser.add_argument("-A", "--async-envs", action="store_true", help="Async parallelization of the environments (otherwise sync)")
arg_parser.add_argument("-R", "--common-rewards", action="store_true", help="Enables common rewards. Each agent will recieve as reward the sum of both agents' rewards")

# Stag Hunt
arg_parser.add_argument("-sf", "--stag-follows", action="store_true", help="Stag follows flag")
arg_parser.add_argument("-sr", "--stag-reward", type=float, default=5.0, help="Stag reward")
arg_parser.add_argument("-mp", "--mauling-punishment", type=float, default=-.5, help="Mauling punishment")
arg_parser.add_argument("-fr", "--forage-reward", type=float, default=1.0, help="Forage reward")
arg_parser.add_argument("-np", "--n-plants", type=int, default=2, help="N of plants")

# DQN
arg_parser.add_argument("-Dse", "--start-epsilon", type=float, default=1.0, help="Start epsilon")
arg_parser.add_argument("-Dee", "--end-epsilon", type=float, default=0.1, help="End epsilon")
arg_parser.add_argument("-Dd", "--time-to-decay", type=float, default=0.7, help="Perc. of time steps to reach epsilon end")
arg_parser.add_argument("-Derb", "--er-buffer-size", type=int, default=10000, help="Experience replay buffer size")
arg_parser.add_argument("-Dbs", "--batch-size", type=int, default=32, help="Batch size")
arg_parser.add_argument("-Dus", "--update-steps", type=int, default=1, help="Update every n steps")

arg_parser.add_argument("-g", "--gamma", type=float, default=.9, help="Gamma discount")
arg_parser.add_argument("-lr", "--learning-rate", type=float, default=3e-4, help="Learning rate")
arg_parser.add_argument("-a", "--alpha", type=float, default=0.6, help="Alpha for prioritized experience replay")
arg_parser.add_argument("-b", "--beta", type=float, default=0.4, help="Beta for prioritized experience replay")
arg_parser.add_argument("-bi", "--beta-iters", type=int, default=100000, help="Beta for prioritized experience replay")


_StoreAction(option_strings=['-bi', '--beta-iters'], dest='beta_iters', nargs=None, const=None, default=100000, type=<class 'int'>, choices=None, required=False, help='Beta for prioritized experience replay', metavar=None)

# Main

## Evaluation Function

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
import torch.nn as nn

class DuelingDQN(nn.Module):
    def __init__(self, input_size, n_actions):
        super(DuelingDQN, self).__init__()
        
        self.feature_net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU()
        )
        
        self.value_head = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.advantages_head = nn.Sequential(
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions)
        )

    def forward(self, x):
        features = self.feature_net(x)
        
        values = self.value_head(features)
        advantages = self.advantages_head(features)
        
        q_vals = values + (advantages - advantages.mean(dim=1, keepdim=True))
        return q_vals


def do_episode(args, llm_model, both_llms=False, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=True):

    ########################
    # CONFIG
    ########################


    ENABLE_GUI = False
    PRINTING_DELAY = 1
    ENABLE_GRAPHIC = False
    PRINT_REWARDS = False

    if args.test:
        ENABLE_GUI = True
        PRINTING_DELAY = .2
        ENABLE_GRAPHIC = True
        PRINT_REWARDS = True

        args.time_steps = 200
        args.episodes = 20

    # Env config
    RUN_AWAY_AFTER_MAUL = False

    PRINTING_TS = 5
    N_STEPS = args.episodes*args.time_steps
    DECAY_STEPS = args.time_to_decay*N_STEPS
    EPSILON_DECAY = (args.start_epsilon-args.end_epsilon)/(DECAY_STEPS)


    env_params = {
        "obs_type": "coords",           # obs_type= image or coords
        "grid_size": (args.width, args.height),
        "screen_size": (1500, 1500),
        "load_renderer": ENABLE_GUI and ENABLE_GRAPHIC,
        "max_timesteps": args.time_steps,
        "forage_quantity": args.n_plants,
        "forage_reward": args.forage_reward,
        "stag_reward": args.stag_reward,
        "mauling_punishment": args.mauling_punishment,
        "stag_follows": args.stag_follows,
        "run_away_after_maul": RUN_AWAY_AFTER_MAUL
    }

    state_shape = (4+2*int(args.n_plants), args.width, args.height)
    looper = LLMStagHuntLooper(env_params, print_rewards=PRINT_REWARDS)
    action_space = looper.get_action_space()
    n_actions = looper.envs.action_space.nvec[0]

    ########################
    # AGENT1
    ########################

    if not both_llms:
        er_buffer_agent1 = PrioritizedExperienceReplayBuffer(state_shape, args.er_buffer_size, start_after=int(0.1*args.er_buffer_size), alpha=args.alpha, beta=args.beta, beta_iters=args.beta_iters)
        dqn_agent1 = DuelingDQN(state_shape[0], n_actions)

        MODEL_NAME = "dqn_mp_-0.5"
        dqn_agent1.load_state_dict(torch.load(f"/kaggle/input/datasets/lorenzozoccadelli/dqn-models/{MODEL_NAME}_ag1.pth"))

        agent1 = DuelingDDQNAgent(dqn_agent1, er_buffer_agent1, args.width, args.height, args.n_plants, action_space, lr=args.learning_rate, batch_size=args.batch_size, update_every_n_steps=args.update_steps, n_envs = args.n_envs, target_update_freq=args.time_steps,
                        start_epsilon=args.start_epsilon, epsilon_decay=EPSILON_DECAY, min_epsilon=args.end_epsilon, gamma=args.gamma, device=device, test=True, linear_decay=True)

    else:
        agent1 = LLMStagHuntAgent(llm_model, args.width, args.height, ask_for_cot=ask_for_cot, ask_for_critique=ask_for_critique, biasing_string=biasing_string, enable_think_tag=enable_think_tag)

    ########################
    # AGENT2
    ########################


    agent2 = LLMStagHuntAgent(llm_model, args.width, args.height, ask_for_cot=ask_for_cot, ask_for_critique=ask_for_critique, biasing_string=biasing_string, enable_think_tag=enable_think_tag)

    ########################
    # Loop
    ########################

    looper.training_loop(agent1, agent2, N_STEPS, common_reward=False, enable_gui=False, ts_to_print=PRINTING_TS)

    return looper, agent1, agent2


## Tests

### Tests 3B

In [14]:

try:
    del llm_model
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

llm_model = Llama.from_pretrained(
		
        repo_id="unsloth/Ministral-3-3B-Reasoning-2512-GGUF",
	    filename="Ministral-3-3B-Reasoning-2512-Q8_0.gguf",
        n_ctx=4096,
        verbose=False,
        n_gpu_layers=-1,
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./Ministral-3-3B-Reasoning-2512-Q8_0.ggu(…):   0%|          | 0.00/3.65G [00:00<?, ?B/s]

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 1.0, mscale_all_dim = 1.0)
llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


In [ ]:
args = arg_parser.parse_args([
    # Generic
    "-s", "-bf", "/", 
    
    # Env
    "-E", "1", "-S", "50", "-e", "1", 
    
    # Stag Hunt
    "-sf", "-sr", "5.0", "-fr", "1.0", "-mp", "-0.5", "-np", "2", 

    "-W", "5", "-H", "5", "-n", "mistral_3_3B_reasoning"
    
])

for i in range(10):
    print(f"Test {i+1}")
    looper, agent1, agent2 = do_episode(args, llm_model, both_llms=True, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False)

    print(looper.log_agents[1].episode_rewards)
    print(looper.log_agents[1].episode_foragings)
    print(looper.log_agents[1].episode_stags)
    print(looper.log_agents[1].episode_maulings)

    agent1.agent_log.save(f"{args.name}_{i+1}_ag1.pkl")
    agent2.agent_log.save(f"{args.name}_{i+1}_ag2.pkl")

In [30]:
args = arg_parser.parse_args([
    # Generic
    "-s", "-bf", "/", 
    
    # Env
    "-E", "1", "-S", "50", "-e", "1", 
    
    # Stag Hunt
    "-sf", "-sr", "5.0", "-fr", "1.0", "-mp", "-0.5", "-np", "2", 

    "-W", "5", "-H", "5", "-n", "mistral_3_3B_reasoning_vs_dqn"
    
])

for i in range(10):
    print(f"Test {i+1}")
    looper, agent1, agent2 = do_episode(args, llm_model, both_llms=False, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False)

    print(looper.log_agents[1].episode_rewards)
    print(looper.log_agents[1].episode_foragings)
    print(looper.log_agents[1].episode_stags)
    print(looper.log_agents[1].episode_maulings)

    agent2.agent_log.save(f"{args.name}_{i+1}_ag2.pkl")

Test 1


/usr/local/lib/python3.12/dist-packages/gymnasium/utils/passive_env_checker.py:245: UserWarning: WARN: The reward returned by `step()` must be a float, int, np.integer or np.floating, actual type: <class 'tuple'>
  logger.warn(


Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 252.15s
[[np.float64(56.0)]]
[[np.int64(6)]]
[[np.int64(10)]]
[[np.int64(0)]]
Test 2
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 246.50s
[[np.float64(28.0)]]
[[np.int64(4)]]
[[np.int64(5)]]
[[np.int64(2)]]
Test 3
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 297.78s
[[np.float64(35.5)]]
[[np.int64(1)]]
[[np.int64(7)]]
[[np.int64(1)]]
Test 4
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 337.72s
[[np.float64(39.5)]]
[[np.int64(1)]]
[[np.int64(8)]]
[[np.int64(3)]]
Test 5
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 352.29s
[[np.float64(42.0)]]
[[np.int64(4)]]
[[np.int64(8)]]
[[np.int64(4

### Tests 8B

In [31]:

try:
    del llm_model
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

llm_model = Llama.from_pretrained(
        repo_id="unsloth/Ministral-3-8B-Reasoning-2512-GGUF",
        filename="Ministral-3-8B-Reasoning-2512-Q8_0.gguf",
        n_ctx=4096,
        verbose=False,
        n_gpu_layers=-1,
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./Ministral-3-8B-Reasoning-2512-Q8_0.ggu(…):   0%|          | 0.00/9.03G [00:00<?, ?B/s]

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 1.0, mscale_all_dim = 1.0)
llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


In [ ]:
args = arg_parser.parse_args([
    # Generic
    "-s", "-bf", "/", 
    
    # Env
    "-E", "1", "-S", "50", "-e", "1", 
    
    # Stag Hunt
    "-sf", "-sr", "5.0", "-fr", "1.0", "-mp", "-0.5", "-np", "2", 

    "-W", "5", "-H", "5", "-n", "mistral_3_8B_reasoning"
    
])

for i in range(10):
    print(f"Test {i+1}")
    looper, agent1, agent2 = do_episode(args, llm_model, both_llms=True, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False)

    print(looper.log_agents[1].episode_rewards)
    print(looper.log_agents[1].episode_foragings)
    print(looper.log_agents[1].episode_stags)
    print(looper.log_agents[1].episode_maulings)

    agent1.agent_log.save(f"{args.name}_{i+1}_ag1.pkl")
    agent2.agent_log.save(f"{args.name}_{i+1}_ag2.pkl")

In [32]:
args = arg_parser.parse_args([
    # Generic
    "-s", "-bf", "/", 
    
    # Env
    "-E", "1", "-S", "50", "-e", "1", 
    
    # Stag Hunt
    "-sf", "-sr", "5.0", "-fr", "1.0", "-mp", "-0.5", "-np", "2", 

    "-W", "5", "-H", "5", "-n", "mistral_3_8B_reasoning_vs_dqn"
    
])

for i in range(10):
    print(f"Test {i+1}")
    looper, agent1, agent2 = do_episode(args, llm_model, both_llms=False, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False)

    print(looper.log_agents[1].episode_rewards)
    print(looper.log_agents[1].episode_foragings)
    print(looper.log_agents[1].episode_stags)
    print(looper.log_agents[1].episode_maulings)

    agent2.agent_log.save(f"{args.name}_{i+1}_ag2.pkl")

Test 1
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 284.10s
[[np.float64(95.5)]]
[[np.int64(3)]]
[[np.int64(19)]]
[[np.int64(5)]]
Test 2
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 317.92s
[[np.float64(82.0)]]
[[np.int64(2)]]
[[np.int64(16)]]
[[np.int64(0)]]
Test 3
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 307.70s
[[np.float64(101.0)]]
[[np.int64(2)]]
[[np.int64(20)]]
[[np.int64(2)]]
Test 4
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 304.76s
[[np.float64(93.5)]]
[[np.int64(5)]]
[[np.int64(18)]]
[[np.int64(3)]]
Test 5
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 287.40s
[[np.float64(86.5)]]
[[np.int64(2)]]
[[np.int64(17)]]


### Tests 14B

In [33]:
try:
    del llm_model
except NameError:
    pass
torch.cuda.empty_cache()
gc.collect()

llm_model = Llama.from_pretrained(
        repo_id="unsloth/Ministral-3-14B-Reasoning-2512-GGUF",
		filename="Ministral-3-14B-Reasoning-2512-Q8_0.gguf",
        n_ctx=4096,
        verbose=False,
        n_gpu_layers=-1,
    )

./Ministral-3-14B-Reasoning-2512-Q8_0.gg(…):   0%|          | 0.00/14.4G [00:00<?, ?B/s]

llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 1.0, mscale_all_dim = 1.0)
llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


In [ ]:
args = arg_parser.parse_args([
    # Generic
    "-s", "-bf", "/", 
    
    # Env
    "-E", "1", "-S", "50", "-e", "1", 
    
    # Stag Hunt
    "-sf", "-sr", "5.0", "-fr", "1.0", "-mp", "-0.5", "-np", "2", 

    "-W", "5", "-H", "5", "-n", "mistral_3_14B_reasoning"
    
])

for i in range(10):
    print(f"Test {i+1}")
    looper, agent1, agent2 = do_episode(args, llm_model, both_llms=True, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False)

    print(looper.log_agents[1].episode_rewards)
    print(looper.log_agents[1].episode_foragings)
    print(looper.log_agents[1].episode_stags)
    print(looper.log_agents[1].episode_maulings)

    agent1.agent_log.save(f"{args.name}_{i+1}_ag1.pkl")
    agent2.agent_log.save(f"{args.name}_{i+1}_ag2.pkl")

In [34]:
args = arg_parser.parse_args([
    # Generic
    "-s", "-bf", "/", 
    
    # Env
    "-E", "1", "-S", "50", "-e", "1", 
    
    # Stag Hunt
    "-sf", "-sr", "5.0", "-fr", "1.0", "-mp", "-0.5", "-np", "2", 

    "-W", "5", "-H", "5", "-n", "mistral_3_14B_reasoning_vs_dqn"
    
])

for i in range(10):
    print(f"Test {i+1}")
    looper, agent1, agent2 = do_episode(args, llm_model, both_llms=False, ask_for_cot=True, ask_for_critique=False, biasing_string="", enable_think_tag=False)

    print(looper.log_agents[1].episode_rewards)
    print(looper.log_agents[1].episode_foragings)
    print(looper.log_agents[1].episode_stags)
    print(looper.log_agents[1].episode_maulings)

    agent2.agent_log.save(f"{args.name}_{i+1}_ag2.pkl")

Test 1
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 657.55s
[[np.float64(100.0)]]
[[np.int64(5)]]
[[np.int64(19)]]
[[np.int64(0)]]
Test 2
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 490.44s
[[np.float64(122.5)]]
[[np.int64(3)]]
[[np.int64(24)]]
[[np.int64(1)]]
Test 3
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 486.88s
[[np.float64(88.5)]]
[[np.int64(0)]]
[[np.int64(18)]]
[[np.int64(3)]]
Test 4
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 625.07s
[[np.float64(106.5)]]
[[np.int64(2)]]
[[np.int64(21)]]
[[np.int64(1)]]
Test 5
Step 5/50
Step 10/50
Step 15/50
Step 20/50
Step 25/50
Step 30/50
Step 35/50
Step 40/50
Step 45/50
Training lasted for 491.33s
[[np.float64(96.0)]]
[[np.int64(2)]]
[[np.int64(19)]